# P109 — La cola a escala

## 1. Título y paper

**Paper:** *The Tail at Scale*  
**Autoría:** Jeffrey Dean, Luiz André Barroso  
**Año y venue:** 2013 · Communications of the ACM, 56(2), 74–80  
**Nivel:** L2 · **Motor:** `cola_larga`  
**Ficha completa:** [`P109_cola_larga`](../../papers/foundational/P109_cola_larga/README.md)

**Hito:** Muestra que con abanico grande la latencia de cola de cada componente se convierte en la latencia típica del sistema completo.

- [doi:10.1145/2408776.2408794](https://doi.org/10.1145/2408776.2408794)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un servicio con un p99 excelente puede producir un sistema lento si la petición del usuario necesita respuesta de cientos de servidores: basta que uno vaya lento para que toda la petición lo vaya, y con cien servidores eso pasa casi siempre.
2. Ejecutar una implementación mínima de la propuesta: Tratar la variabilidad de latencia como propiedad de diseño y no como ruido. Técnicas de tolerancia a la cola —peticiones de cobertura, cancelación cruzada, micro-particionado, réplicas selectivas— que reducen la cola sin eliminar sus causas.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P107


## 4. Intuición

Cada uno de tus cien servidores tiene un p99 excelente. La petición del usuario necesita respuesta de los cien. Y entonces la **mediana** de tu sistema es peor que la cola de cualquiera de ellos, porque basta que uno vaya lento.


## 5. Concepto mínimo

```text
Si cada servidor se para con probabilidad p, la probabilidad de que
ALGUNO de n se pare es  1 − (1−p)ⁿ

    p = 1 %,  n = 100   →   63 %
    p = 1 %,  n = 1000  →   99,996 %

Con abanico grande, lo raro es lo normal.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('cola_larga', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuál es el p99 de un servidor solo?
2. ¿Cuál es el p50 de una petición que necesita 100 servidores?
3. ¿Cuánto ayuda pedir a una segunda réplica cuando la primera tarda?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('cola_larga', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('cola_larga', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Un servidor solo tiene un p99 de **70,4 ms**. Con 100 servidores en paralelo, el p50 de la petición completa sube a **947 ms**: la mediana del conjunto es trece veces peor que la cola de uno solo. Las peticiones de cobertura bajan esa mediana a **69,7 ms** —un factor de 13,6×— y el p99 apenas se mueve, porque a veces fallan las dos réplicas.


## 10. Comentario pedagógico

Ese último detalle es el honesto: la cobertura **recorta** la cola, no la elimina. Y tiene un coste de tráfico que hay que presupuestar. La conclusión operativa del artículo es que en sistemas con abanico grande la latencia de cola no es un detalle de rendimiento: es la latencia que ve el usuario, y hay que diseñarla.


## 11. Error o anti-patrón deliberado

Anti-patrón: optimizar la media cuando el problema es la cola.


In [ ]:
print('Bajar el p50 de un servidor de 50 a 40 ms no cambia casi nada.')
print('Con 100 servidores, lo que ve el usuario lo determina el mas lento de los 100.')
print('Optimizar la media es trabajar sobre la parte que no manda.')

## 12. Corrección

Dónde está el problema y dónde no:


In [ ]:
r = run_paper_lab('cola_larga', seed=7)['result']
for f in r['escalado_por_abanico']:
    print(f"  {f['servidores']:>5} servidores  p50={f['p50_ms']:<8} p99={f['p99_ms']:<8}"
          f"  P(alguno lento)={f['prob_alguno_lento']}")
print()
print('con cobertura :', r['con_peticiones_de_cobertura_100_servidores'])

## 13. Desafío guiado

Calcula la probabilidad de que alguno de 1 000 servidores se pare si cada uno lo hace el 0,1 % de las veces, y explica qué implica para el diseño.


In [ ]:
r = run_paper_lab('cola_larga', seed=3)['result']
show(r)

## 14. Desafío autónomo

Mide el p50 y el p99 de un servicio tuyo que haga llamadas en paralelo. Calcula cuántas hace y estima la contribución de la cola de cada dependencia.


## 15. Evidencia de aprendizaje

Guarda la tabla de escalado por abanico y tu cálculo de la probabilidad de cola para tu propio sistema.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P109_cola_larga/README.md) · evaluación formal: [`assessments/papers/P109_cola_larga.md`](../../assessments/papers/P109_cola_larga.md)


## 16. Cierre

El sistema ya responde rápido y sobrevive a las particiones. Ahora el modelo: qué pasa cuando el mundo cambia y él no.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
